# Milestone 0 — Smoke Test (revised)

**Project**: PBT Meta-Controller (Hierarchical PBT, Phase 1)
**Goal**: Verify all components install and connect on Colab L4 — **before** committing scope of M1+

This notebook tests **5 assumptions** that the rest of the project depends on. Each cell prints `[PASS]` or `[FAIL]`.

| # | Assumption | If FAIL → |
|---|---|---|
| 1 | Colab provides L4 GPU with ≥20 GB | Switch runtime |
| 2 | Genesis 0.4.3 installs cleanly | Check version pin |
| 3 | Genesis can render Go2 headlessly via gs.Camera | Need viewer fallback |
| 4 | Ollama installs + runs as background service | May need different VLM engine |
| 5 | Qwen2.5-VL-3B pulls + responds to image+text | Verify model tag |

**Time budget**: 20-30 min total

**Setup before running**:
1. Runtime → Change runtime type → **L4 GPU** (NOT T4)
2. Run cells in order — do not skip

## Test 1 — GPU availability

In [ ]:
# Test 1: Verify L4 GPU with ≥20 GB
import subprocess
import sys

print("=" * 60)
print("TEST 1: GPU availability")
print("=" * 60)

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, check=True
    )
    gpu_info = result.stdout.strip()
    print(f"GPU: {gpu_info}")

    parts = gpu_info.split(",")
    name = parts[0].strip()
    mem_mib = int(parts[1].strip().split()[0])
    mem_gb = mem_mib / 1024

    if mem_gb >= 20:
        print(f"[PASS] {name} with {mem_gb:.1f} GB")
    elif mem_gb >= 14:
        print(f"[WARN] {name} with {mem_gb:.1f} GB — tight, close other notebooks")
    else:
        print(f"[FAIL] Only {mem_gb:.1f} GB — switch to L4 runtime")
        sys.exit(1)
except FileNotFoundError:
    print("[FAIL] nvidia-smi not found — Runtime → Change runtime type → L4 GPU")
    sys.exit(1)

## Test 2 — Install Genesis 0.4.3

This cell takes 1-3 minutes.

In [ ]:
# Test 2: Install Genesis 0.4.3
import subprocess
import time

print("=" * 60)
print("TEST 2: Install Genesis 0.4.3")
print("=" * 60)

t0 = time.time()
result = subprocess.run(
    ["pip", "install", "-q", "genesis-world==0.4.3"],
    capture_output=True, text=True
)
elapsed = time.time() - t0

if result.returncode == 0:
    print(f"[PASS] Installed in {elapsed:.0f}s")
else:
    print(f"[FAIL] Install failed in {elapsed:.0f}s")
    print(result.stderr[-500:])

In [ ]:
# Test 2b: Verify import
print("=" * 60)
print("TEST 2b: Import Genesis")
print("=" * 60)

try:
    import genesis as gs
    version = gs.__version__ if hasattr(gs, '__version__') else 'unknown'
    print(f"[PASS] genesis imported, version = {version}")
except Exception as e:
    print(f"[FAIL] {type(e).__name__}: {e}")

## Test 3 — Genesis renders Go2 headlessly

Colab has no display, so `show_viewer=True` won't work. We use `gs.Camera` to render frames as numpy arrays.

**Note**: First build is slow (~2 min) due to taichi JIT compilation. Subsequent builds are fast.

In [ ]:
# Test 3: Init Genesis, spawn Go2, render 1 frame
import time
import numpy as np

print("=" * 60)
print("TEST 3: Genesis render Go2 headlessly")
print("=" * 60)

import genesis as gs

try:
    print("Initializing Genesis on GPU...")
    t0 = time.time()
    gs.init(backend=gs.gpu, precision="32", logging_level="warning")
    print(f"  init: {time.time() - t0:.1f}s")

    print("Creating scene + Go2 + camera...")
    t0 = time.time()
    scene = gs.Scene(
        sim_options=gs.options.SimOptions(dt=0.02, substeps=2),
        viewer_options=gs.options.ViewerOptions(
            camera_pos=(2.5, 0.0, 1.5),
            camera_lookat=(0.0, 0.0, 0.5),
            camera_fov=40,
        ),
        show_viewer=False,  # CRITICAL on Colab
    )

    plane = scene.add_entity(gs.morphs.Plane())

    go2 = scene.add_entity(
        gs.morphs.URDF(
            file="urdf/go2/urdf/go2.urdf",
            pos=(0.0, 0.0, 0.42),
        ),
    )

    cam = scene.add_camera(
        res=(640, 480),
        pos=(2.5, 0.0, 1.5),
        lookat=(0.0, 0.0, 0.5),
        fov=40,
        GUI=False,
    )

    scene.build(n_envs=1)
    print(f"  build: {time.time() - t0:.1f}s")

    print("Stepping + rendering...")
    t0 = time.time()
    scene.step()
    rgb_array, depth, seg, normal = cam.render(rgb=True)
    elapsed = time.time() - t0
    print(f"  step+render: {elapsed:.2f}s")

    if rgb_array is not None and rgb_array.shape == (480, 640, 3):
        print(f"[PASS] Rendered frame, shape={rgb_array.shape}, dtype={rgb_array.dtype}")
        import pickle
        with open("/tmp/m0_test_frame.pkl", "wb") as f:
            pickle.dump(rgb_array, f)
    else:
        shape = rgb_array.shape if rgb_array is not None else None
        print(f"[FAIL] Unexpected frame shape: {shape}")

except Exception as e:
    import traceback
    print(f"[FAIL] {type(e).__name__}: {e}")
    traceback.print_exc()

In [ ]:
# Test 3b: Display the rendered frame inline
import pickle
import matplotlib.pyplot as plt

try:
    with open("/tmp/m0_test_frame.pkl", "rb") as f:
        rgb = pickle.load(f)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(rgb)
    ax.set_title("Go2 standing — rendered headlessly via gs.Camera")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"[PASS] Frame displayed, range [{rgb.min()}, {rgb.max()}]")
except FileNotFoundError:
    print("[FAIL] No frame to display — Test 3 failed")
except Exception as e:
    print(f"[FAIL] {e}")

## Test 4 — Install Ollama + start service

Ollama is a process. We install via the official curl script then start it in background.

**Critical fix**: must install `zstd + pciutils + lshw` apt deps first, and use `!` shell magic (not `subprocess.run`) so sudo works in Colab.

In [ ]:
# Test 4a: Install Ollama
print("=" * 60)
print("TEST 4a: Install Ollama")
print("=" * 60)

# Step 1: apt deps (zstd is required by recent Ollama installer)
print("Installing apt deps (zstd, pciutils, lshw)...")
!sudo apt-get install -y zstd pciutils lshw > /tmp/apt.log 2>&1
print("  done")

# Step 2: Official Ollama install script
print("Installing Ollama (1-2 min)...")
!curl -fsSL https://ollama.com/install.sh | sh

# Step 3: Set CUDA lib path for Ollama
import os
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia'

# Step 4: Verify
import shutil
ollama_path = shutil.which("ollama")
if ollama_path:
    print(f"[PASS] Ollama installed at {ollama_path}")
    !ollama --version
else:
    print("[FAIL] Ollama binary not found")
    !tail -20 /tmp/apt.log

In [ ]:
# Test 4b: Start Ollama service
import subprocess
import time
import os

print("=" * 60)
print("TEST 4b: Start Ollama service")
print("=" * 60)

os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia'

# Kill any existing ollama process
!pkill -f "ollama serve" 2>/dev/null
time.sleep(1)

# Start ollama serve in background (& makes it background)
get_ipython().system_raw('nohup ollama serve > /tmp/ollama.log 2>&1 &')

# Wait for service to be ready
print("Waiting for service to start...")
ready = False
for i in range(20):
    time.sleep(1)
    try:
        result = subprocess.run(
            ["curl", "-s", "http://localhost:11434/api/version"],
            capture_output=True, text=True, timeout=2
        )
        if result.returncode == 0 and result.stdout:
            print(f"[PASS] Service ready after {i+1}s")
            print(f"  version: {result.stdout.strip()}")
            ready = True
            break
    except Exception:
        pass

if not ready:
    print("[FAIL] Service did not start within 20s")
    !tail -30 /tmp/ollama.log

## Test 5 — Pull Qwen2.5-VL-3B + test image+text response

This is the **biggest unknown** in M0:
1. Is `qwen2.5vl:3b` the correct Ollama tag?
2. Does it download successfully? (~3 GB)
3. Does it accept image+text input via Ollama Python client?
4. What's the actual latency on L4?

In [ ]:
# Test 5a: Install ollama Python client
import subprocess
print("=" * 60)
print("TEST 5a: Install ollama Python client")
print("=" * 60)

result = subprocess.run(
    ["pip", "install", "-q", "ollama"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("[PASS] ollama Python client installed")
    import ollama
    v = ollama.__version__ if hasattr(ollama, '__version__') else 'unknown'
    print(f"  client version: {v}")
else:
    print(f"[FAIL] {result.stderr[-300:]}")

In [ ]:
# Test 5b: Pull qwen2.5vl:3b (~3 GB)
import subprocess
import time

print("=" * 60)
print("TEST 5b: Pull qwen2.5vl:3b")
print("=" * 60)

MODEL_TAG = "qwen2.5vl:3b"

t0 = time.time()
result = subprocess.run(
    ["ollama", "pull", MODEL_TAG],
    capture_output=True, text=True
)
elapsed = time.time() - t0

if result.returncode == 0:
    print(f"[PASS] Pulled '{MODEL_TAG}' in {elapsed:.0f}s")
else:
    print(f"[FAIL] Could not pull '{MODEL_TAG}'")
    print(f"stderr: {result.stderr[-500:]}")

In [ ]:
# Test 5c: Send image + text to Qwen
import time
import pickle
from io import BytesIO
from PIL import Image
import ollama

print("=" * 60)
print("TEST 5c: Image + text inference")
print("=" * 60)

MODEL_TAG = "qwen2.5vl:3b"

try:
    with open("/tmp/m0_test_frame.pkl", "rb") as f:
        rgb = pickle.load(f)
except FileNotFoundError:
    print("[FAIL] No test frame from Test 3")
else:
    img = Image.fromarray(rgb.astype("uint8"))
    buf = BytesIO()
    img.save(buf, format="PNG")
    img_bytes = buf.getvalue()
    print(f"  image size: {len(img_bytes) / 1024:.1f} KB")

    # First call (cold start — model load)
    print("Sending image + prompt (first call, cold start)...")
    t0 = time.time()
    try:
        response = ollama.chat(
            model=MODEL_TAG,
            messages=[{
                "role": "user",
                "content": "What do you see in this image? Respond in 1 sentence.",
                "images": [img_bytes],
            }],
        )
        latency_cold = time.time() - t0
        reply = response["message"]["content"]
        print(f"  cold latency: {latency_cold:.1f}s")
        print(f"  reply: {reply[:200]}")

        # Second call (warm)
        print()
        print("Sending second prompt (warm)...")
        t0 = time.time()
        response2 = ollama.chat(
            model=MODEL_TAG,
            messages=[{
                "role": "user",
                "content": "Is there a robot in this image? Yes or no.",
                "images": [img_bytes],
            }],
        )
        latency_warm = time.time() - t0
        reply2 = response2["message"]["content"]
        print(f"  warm latency: {latency_warm:.1f}s")
        print(f"  reply: {reply2[:200]}")

        print()
        if latency_warm < 5.0:
            print(f"[PASS] Warm latency {latency_warm:.1f}s — fast enough for slow loop")
        elif latency_warm < 10.0:
            print(f"[WARN] Warm latency {latency_warm:.1f}s — slow loop will be < 1 Hz")
        else:
            print(f"[FAIL] Warm latency {latency_warm:.1f}s — too slow")

    except Exception as e:
        import traceback
        print(f"[FAIL] {type(e).__name__}: {e}")
        traceback.print_exc()

## Summary

In [ ]:
# Final summary
import subprocess
import psutil

print("=" * 60)
print("MILESTONE 0 SUMMARY")
print("=" * 60)

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True
    )
    used, total = [int(x.strip()) for x in result.stdout.strip().split(",")]
    used_gb = used / 1024
    total_gb = total / 1024
    free_gb = total_gb - used_gb
    print(f"GPU memory:  {used_gb:.1f} / {total_gb:.1f} GB used ({free_gb:.1f} GB free)")
except Exception as e:
    print(f"GPU memory: unable to read ({e})")

mem = psutil.virtual_memory()
print(f"System RAM:  {mem.used / 1e9:.1f} / {mem.total / 1e9:.1f} GB used")

print()
print("If all tests PASS → ready for M1 (m1_train_skill.ipynb)")
print("If any FAIL → paste output to chat for diagnosis")

## Next steps

1. **All PASS** → open `m1_train_skill.ipynb` in same Colab session (Genesis stays loaded)
2. **Any FAIL** → paste failing cell output to chat
3. Things to report regardless:
   - Cold latency of Qwen first call
   - Warm latency of subsequent calls
   - GPU memory after all tests